# AMD/NVIDIA SEC 提交索引当前目录

固定两个官方 CIK JSON，核对发行人、最近 1000+ 文件和官方历史分卷清单。

In [1]:
import json,hashlib,subprocess,time,tarfile,tempfile,shlex
from pathlib import Path
from datetime import datetime,timezone
from src.data.nas_market_data import _settings,nas_api_call
root=Path.cwd();card=root/'research/stage1_amd_nvda_sec_guidance_original_probe_20261023';raw=root/'data/snapshots/stage1_amd_nvda_sec_guidance_original_probe_20261023';plan=json.loads((card/'main_submissions_plan.json').read_text())
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and sha(card/'result.json')==plan['parent_anchor_result_sha256'] and not (card/'main_submissions_state.json').exists()
state={'status':'started','GET_attempts':0,'model_fits':0,'native_accounts':0,'market_outcome_reads':0};(card/'main_submissions_state.json').write_text(json.dumps(state,indent=2)+'\n');receipts=[];checks=[];failure=None
for spec in plan['exact_requests']:
 name=spec['issuer'].lower();body=raw/f'{name}_main_submissions.json';headers=raw/f'{name}_main_submissions.headers';state['GET_attempts']+=1;(card/'main_submissions_state.json').write_text(json.dumps(state,indent=2)+'\n');started=datetime.now(timezone.utc).isoformat()
 cmd=['curl','--http1.1','--noproxy','*','--connect-timeout','10','--max-time','60','--max-filesize','8000000','--fail-with-body','-sS','-L','-D',str(headers),'-o',str(body),'-A',plan['request']['user_agent'],'-w','%{http_code} %{url_effective}',spec['url']]
 try:
  run=subprocess.run(cmd,capture_output=True,text=True,timeout=75);parts=run.stdout.rsplit(' ',1);status=int(parts[0]) if len(parts)==2 and parts[0].isdigit() else None
  receipt={'issuer':spec['issuer'],'url':spec['url'],'requested_at_utc':started,'received_at_utc':datetime.now(timezone.utc).isoformat(),'curl_exit_code':run.returncode,'http_status':status,'final_url':parts[-1] if parts else None,'stderr':run.stderr[:250]}
 except Exception as exc:receipt={'issuer':spec['issuer'],'url':spec['url'],'requested_at_utc':started,'received_at_utc':datetime.now(timezone.utc).isoformat(),'exception':str(exc)[:250]}
 receipt.update(raw_sha256=sha(body) if body.exists() else None,raw_bytes=body.stat().st_size if body.exists() else 0,headers_sha256=sha(headers) if headers.exists() else None);receipts.append(receipt);(card/'main_submissions_receipts.json').write_text(json.dumps(receipts,indent=2)+'\n')
 if receipt.get('curl_exit_code')!=0 or receipt.get('http_status')!=200 or not body.exists():failure=f'{name}_transport';break
 try:
  j=json.loads(body.read_text());assert str(j['cik']).lstrip('0')==str(int(spec['CIK']));s=j['filings']['recent'];keys=('accessionNumber','filingDate','form','acceptanceDateTime','primaryDocument');n=len(s['accessionNumber']);assert n>=1000 and all(len(s[k])==n for k in keys) and len(set(s['accessionNumber']))==n
  catalog=j['filings'].get('files',[]);assert all(x['name'].startswith('CIK'+spec['CIK']+'-submissions-') and x['name'].endswith('.json') and x['filingCount']>0 for x in catalog)
  checks.append({'issuer':spec['issuer'],'CIK':spec['CIK'],'raw_sha256':sha(body),'recent_rows':n,'recent_first_filing':s['filingDate'][0],'recent_last_filing':s['filingDate'][-1],'older_catalog':catalog});(card/'main_submissions_checks.json').write_text(json.dumps(checks,indent=2)+'\n')
 except Exception as exc:failure=f'{name}_schema_{type(exc).__name__}_{str(exc)[:120]}';break
 time.sleep(plan['request']['seconds_between_requests'])
files=[{'name':x.name,'sha256':sha(x),'bytes':x.stat().st_size} for x in sorted(raw.iterdir()) if x.name.endswith(('.json','.headers')) and 'main_submissions' in x.name];manifest={'plan_sha256':sha(card/'main_submissions_plan.json'),'files':files,'receipts_sha256':sha(card/'main_submissions_receipts.json')};(card/'main_submissions_manifest.json').write_text(json.dumps(manifest,indent=2)+'\n')
settings=_settings(root)
with tempfile.TemporaryDirectory(prefix='amd-nvda-submissions-') as td:
 archive=Path(td)/'source.tar.gz'
 with tarfile.open(archive,'w:gz') as t:
  for x in files:t.add(raw/x['name'],arcname='raw/'+x['name'])
  for name in ('main_submissions_plan.json','main_submissions_receipts.json','main_submissions_manifest.json'):t.add(card/name,arcname=name)
 digest=sha(archive);a=nas_api_call(origin=settings['origin'],token=settings['token'],method='PUT',path='/api/research/artifacts/'+digest,upload=archive,executor_id='soxl-minute-research')['artifact'];assert a['sha256']==digest and a['size_bytes']==archive.stat().st_size
 remote=settings['artifact_root'].rstrip('/')+'/'+digest[:2]+'/'+digest[2:4]+'/'+digest
 with tempfile.TemporaryDirectory(prefix='amd-nvda-submissions-restore-') as rd:
  restored=Path(rd)/'restore.tar.gz'
  with restored.open('wb') as out:run=subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],'cat '+shlex.quote(remote)],stdout=out,stderr=subprocess.PIPE,timeout=120)
  assert run.returncode==0 and sha(restored)==digest
  with tarfile.open(restored,'r:gz') as t:
   for x in files:
    m=t.extractfile('raw/'+x['name']);assert m and hashlib.sha256(m.read()).hexdigest()==x['sha256']
result={'status':'two_issuer_SEC_main_submissions_catalogs_passed_history_pending' if not failure and len(checks)==2 else 'bounded_two_issuer_SEC_main_submissions_partial_or_failed','failure':failure,'plan_sha256':sha(card/'main_submissions_plan.json'),'GET_attempts':state['GET_attempts'],'checks':checks,'NAS_archive_sha256':digest,'NAS_remote_and_scoped_restore_verified':True,'full_2011_2026_history_qualified':False,'market_outcome_reads':0,'model_fits':0,'native_accounts':0};(card/'main_submissions_result.json').write_text(json.dumps(result,indent=2)+'\n');state.update(status='completed' if not failure else 'partial_or_failed',NAS_archive_sha256=digest);(card/'main_submissions_state.json').write_text(json.dumps(state,indent=2)+'\n');print({'status':result['status'],'failure':failure,'GETs':state['GET_attempts'],'catalogs':[(x['issuer'],len(x['older_catalog'])) for x in checks],'NAS':digest})


{'status': 'two_issuer_SEC_main_submissions_catalogs_passed_history_pending', 'failure': None, 'GETs': 2, 'catalogs': [('NVDA', 1), ('AMD', 2)], 'NAS': '4506b7bd2a6e53625b2fd3193efc9aef9e20af55696ad7e80b0a061ecfb0f8ee'}


## 中文结论

AMD/NVIDIA 当前 SEC 申报目录通过 CIK、数组、历史分卷声明和 NAS 验证。 完整决定见 `conclusion.md`。